In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

In [2]:
# 1. Definimos una única ruta raíz para la Fact Table
GOLD_ZONE_ROOT = "../DATA/Gold/COVID_FACT"

# 2. Lista de archivos de origen
ORIGIN_FILES = [
    "../DATA/Silver/COVID19MEXICO2020/COVID19MEXICO2020.csv",
    "../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv",
    "../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv",
    "../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv",
    "../DATA/Silver/COVID19MEXICO2024/COVID19MEXICO2024.csv"
]

DIMENSIONS = {
    'DIM_Geografico_informacion_paciente': ['ENTIDAD_UM', 'ENTIDAD_NAC'],
    'DIM_Geografico_residencia': ['ENTIDAD_RES', 'MUNICIPIO_RES'],
    'DIM_Geografico_Nacionalidad': ['NACIONALIDAD', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN'],
    'DIM_Descripcion_del_paciente': ['SEXO', 'EDAD', 'TIPO_PACIENTE'],
    'DIM_Indigena': ['HABLA_LENGUA_INDIG', 'INDIGENA', 'MIGRANTE'],
    'DIM_Comorbilidades_Respiratorias': ['INTUBADO', 'NEUMONIA', 'EPOC', 'ASMA', 'TABAQUISMO'],
    'DIM_Comorbilidades_de_presion': ['DIABETES', 'INMUSUPR', 'HIPERTENSION', 'CARDIOVASCULAR', 'OBESIDAD'],
    'DIM_Otras_caracteristicas_medicas': ['EMBARAZO', 'RENAL_CRONICA', 'OTRA_COM'],
    'DIM_Ubicacion_de_laboratorio': ['ORIGEN', 'SECTOR', 'OTRO_CASO', 'UCI', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION'],
    'DIM_Antigeno': ['TOMA_MUESTRA_ANTIGENO', 'RESULTADO_ANTIGENO'],
    'DIM_Datos_de_laboratorio': ['TOMA_MUESTRA_LAB', 'RESULTADO_LAB', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU']
}

DIMENSIONS_PATH = "../TRANSFORM/dimensiones"

In [3]:
def load_dimensions_in_memory():
    """Carga las tablas de dimensiones físicas, conservando solo el ID y las columnas clave para el Join."""
    dict_dims = {}
    print("Cargando dimensiones en memoria RAM...")
    
    for name_dim, columns in DIMENSIONS.items():
        csv_path = os.path.join(DIMENSIONS_PATH, f"{name_dim}.csv")
        if os.path.exists(csv_path):
            df_dim = pd.read_csv(csv_path, low_memory=False)
            id_column = f'ID_{name_dim.upper()}'
            necesary_column = [id_column] + columns
            dict_dims[name_dim] = df_dim[necesary_column]
        else:
            print(f"[Advertencia] No se encontró la dimensión {csv_path}")
            
    return dict_dims

In [4]:
def process_parquet_facts(dict_dims):
    """Itera sobre los CSV masivos, cruza con dimensiones, y escribe de forma particionada (Hive) en Parquet."""
    
    for origin_file in ORIGIN_FILES:
        if not os.path.exists(origin_file):
            print(f"[Saltando] Archivo no encontrado: {origin_file}")
            continue
            
        print(f"\nIniciando transformación y particionamiento de: {origin_file}")
        
        ok_rows = 0
        batch_iterator = pd.read_csv(origin_file, chunksize=250000, low_memory=False, encoding='latin1')
        
        for chunk in batch_iterator:
            ok_chunck = chunk.copy()
            remove_columns = []
            
            # 1. Sustitución de Llaves (Left Joins)
            for name_dim, columns_cruce in DIMENSIONS.items():
                if name_dim not in dict_dims: continue
                df_dim = dict_dims[name_dim]
                cols_presents = [c for c in columns_cruce if c in ok_chunck.columns]
                
                if len(cols_presents) == len(columns_cruce):
                    ok_chunck = pd.merge(ok_chunck, df_dim, on=columns_cruce, how='left')
                    remove_columns.extend(columns_cruce)
            
            remove_columns = list(set(remove_columns))
            ok_chunck.drop(columns=remove_columns, inplace=True, errors='ignore')
            
            # 2. Optimización de Tipos de Datos (IDs a Int64)
            for_to_int = [col for col in ok_chunck.columns if col.startswith('ID_DIM_')]
            for col in for_to_int:
                ok_chunck[col] = ok_chunck[col].astype('Int64')
                
            # 3. Lógica de Particionamiento: Extracción de Año y Mes
            # Utilizamos FECHA_INGRESO como pivote temporal analítico
            if 'FECHA_INGRESO' in ok_chunck.columns:
                # Convertimos a datetime manejando posibles errores o fechas nulas
                fechas = pd.to_datetime(ok_chunck['FECHA_INGRESO'], errors='coerce')
                
                # Extraemos el año y mes como strings. Los nulos se envían a una partición 'unknown'
                ok_chunck['year'] = fechas.dt.year.fillna(9999).astype(int).astype(str)
                # Aplicamos zfill(2) para que los meses queden como 01, 02, etc. Garantiza orden lexicográfico.
                ok_chunck['month'] = fechas.dt.month.fillna(99).astype(int).astype(str).str.zfill(2)
            else:
                # Fallback si el chunk no tiene fecha (garantiza la estabilidad del proceso)
                ok_chunck['year'] = 'unknown'
                ok_chunck['month'] = 'unknown'

            # 4. Escritura en Parquet con estructura Hive
            table_pyarrow = pa.Table.from_pandas(ok_chunck)
            
            pq.write_to_dataset(
                table_pyarrow,
                root_path=GOLD_ZONE_ROOT,
                partition_cols=['year', 'month'],
                compression='snappy'
                # Por defecto, PyArrow genera archivos con UUIDs, previniendo sobrescrituras
            )
            
            ok_rows += len(ok_chunck)
            print(f"  -> Lote procesado y particionado... ({ok_rows} filas distribuidas)")
            
        print(f"[Éxito] Archivo {origin_file} procesado e integrado al Data Lake local.")

In [5]:
memory_dimensions = load_dimensions_in_memory()
process_parquet_facts(memory_dimensions)

Cargando dimensiones en memoria RAM...

Iniciando transformación y particionamiento de: ../DATA/Silver/COVID19MEXICO2020/COVID19MEXICO2020.csv
  -> Lote procesado y particionado... (99990 filas distribuidas)
[Éxito] Archivo ../DATA/Silver/COVID19MEXICO2020/COVID19MEXICO2020.csv procesado e integrado al Data Lake local.

Iniciando transformación y particionamiento de: ../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv
  -> Lote procesado y particionado... (101342 filas distribuidas)
[Éxito] Archivo ../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv procesado e integrado al Data Lake local.

Iniciando transformación y particionamiento de: ../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv
  -> Lote procesado y particionado... (99986 filas distribuidas)
[Éxito] Archivo ../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv procesado e integrado al Data Lake local.

Iniciando transformación y particionamiento de: ../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv
  -> Lote 